In [2]:
import pandas as pd
from pathlib import Path
from datetime import datetime

In [7]:
DATA_DIR = Path("data")
US_PATH = DATA_DIR / "USvideos.csv"
GB_PATH = DATA_DIR / "GBvideos.csv"
CATEGORY_PATH = DATA_DIR / "categories.csv"

In [9]:
COUNTRY_PATHS = {
    "US": US_PATH,
    "GB": GB_PATH,
}

MAX_ROWS = None             # Set to 1000 for a smaller test
VIEW_THRESHOLD = 1_000_000
LIKE_RATIO_THRESHOLD = 0.04  # Likes / views
TOP_CHANNELS = 15

print(f"""
========================================
TrendPulse | Part 1: Core Python
Run time: {datetime.now():%Y-%m-%d %H:%M:%S}
Countries: {', '.join(COUNTRY_PATHS.keys())}
View threshold: {VIEW_THRESHOLD:,}
========================================
""")



TrendPulse | Part 1: Core Python
Run time: 2026-10-26 15:06:55
Countries: US, GB
View threshold: 1,000,000



In [10]:

def load_country(path, country, max_rows=None):
    """Load a country's trending-video CSV as a list of dictionaries."""
    df = pd.read_csv(path, nrows=max_rows)
    df["country"] = country
    return df.to_dict("records")


# Load the category mapping
category_names = {}

if CATEGORY_PATH.exists():
    category_df = pd.read_csv(CATEGORY_PATH)

    category_names = {
        str(row["category_id"]): row["category_name"]
        for _, row in category_df.iterrows()
    }

# Load available country files
records_by_country = {}

for country, path in COUNTRY_PATHS.items():
    if path.exists():
        records_by_country[country] = load_country(
            path, country, MAX_ROWS
        )
        print(f"{country}: {len(records_by_country[country]):,} records")
    else:
        print(f"Skipping {country}: file not found")

# Combine records into one list
records = []

for country_records in records_by_country.values():
    records.extend(country_records)

print("Total records:", len(records))
print("Category examples:", list(category_names.items())[:5])
print("First record:", records[0] if records else "No data loaded")


US: 16,080 records
GB: 15,075 records
Total records: 31155
Category examples: [('1', 'Film & Animation'), ('2', 'Autos & Vehicles'), ('10', 'Music'), ('15', 'Pets & Animals'), ('17', 'Sports')]
First record: {'video_id': 'g_N-w8LeAi8', 'trending_date': '17.14.11', 'title': 'Honest Football Prank LEAKED', 'channel_title': 'QuietSpeech10', 'category_id': 22, 'publish_time': '2017-11-14T07:04:28.000Z', 'tags': 'Monsoon|Robots', 'views': 48555, 'likes': 2189.0, 'dislikes': 180, 'comment_count': 136.0, 'thumbnail_link': 'https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about chess. Business enquiries in the channel about page.', 'country': 'US'}


In [11]:

from datetime import datetime

failures = {
    "views": 0,
    "likes": 0,
    "dislikes": 0,
    "comment_count": 0,
    "trending_date": 0,
    "video_id": 0,
    "title": 0,
}

clean_records = []
dropped_records = 0

for record in records:
    cleaned = record.copy()

    # Video ID: drop if missing or blank.
    try:
        video_id = str(cleaned["video_id"]).strip()
        if not video_id or video_id.lower() == "nan":
            raise ValueError("Missing video ID")
        cleaned["video_id"] = video_id
    except (KeyError, TypeError, ValueError):
        failures["video_id"] += 1
        dropped_records += 1
        continue  # Cannot reliably identify this record.

    # Title: drop if missing or blank.
    try:
        title = str(cleaned["title"]).strip()
        if not title or title.lower() == "nan":
            raise ValueError("Missing title")
        cleaned["title"] = title
    except (KeyError, TypeError, ValueError):
        failures["title"] += 1
        dropped_records += 1
        continue  # A title is required for title analysis.

    # Views: essential numeric field.
    try:
        views = int(cleaned["views"])
        if views < 0:
            raise ValueError("Negative views")
        cleaned["views"] = views
    except (KeyError, TypeError, ValueError, OverflowError):
        failures["views"] += 1
        dropped_records += 1
        continue  # Cannot analyze popularity without valid views.

    # Likes: substitute zero if invalid.
    try:
        likes = int(cleaned["likes"])
        if likes < 0:
            raise ValueError("Negative likes")
        cleaned["likes"] = likes
    except (KeyError, TypeError, ValueError, OverflowError):
        failures["likes"] += 1
        cleaned["likes"] = 0  # Simplifying assumption; log the failure.

    # Dislikes: substitute zero if invalid.
    try:
        dislikes = int(cleaned["dislikes"])
        if dislikes < 0:
            raise ValueError("Negative dislikes")
        cleaned["dislikes"] = dislikes
    except (KeyError, TypeError, ValueError, OverflowError):
        failures["dislikes"] += 1
        cleaned["dislikes"] = 0  # Missing is not necessarily zero.

    # Comment count: substitute zero if invalid.
    try:
        comments = int(cleaned["comment_count"])
        if comments < 0:
            raise ValueError("Negative comment count")
        cleaned["comment_count"] = comments
    except (KeyError, TypeError, ValueError, OverflowError):
        failures["comment_count"] += 1
        cleaned["comment_count"] = 0

    # Date format: yy.dd.mm, e.g. 17.14.11 = 14 Nov 2017.
    try:
        date_text = str(cleaned["trending_date"]).strip()
        cleaned["trending_date"] = datetime.strptime(
            date_text, "%y.%d.%m"
        )
    except (KeyError, TypeError, ValueError):
        failures["trending_date"] += 1
        dropped_records += 1
        continue  # Invalid dates cannot be used in date analysis.

    clean_records.append(cleaned)

print("Original records:", len(records))
print("Clean records:", len(clean_records))
print("Dropped records:", dropped_records)
print("Conversion failures:", failures)


Original records: 31155
Clean records: 31155
Dropped records: 0
Conversion failures: {'views': 0, 'likes': 8, 'dislikes': 0, 'comment_count': 24, 'trending_date': 0, 'video_id': 0, 'title': 0}


In [12]:

# 1. List of dictionaries
print("Record example:", clean_records[0] if clean_records else None)

# 2. Tuples: (video_id, trending_date, views, likes)
video_tuples = []

for record in clean_records:
    video_tuples.append((
        record["video_id"],
        record["trending_date"],
        record["views"],
        record["likes"],
    ))

print("First tuples:", video_tuples[:5])

# 3. Sets of unique video IDs per country
video_ids_by_country = {}

for country in records_by_country:
    video_ids_by_country[country] = set()

for record in clean_records:
    video_ids_by_country[record["country"]].add(record["video_id"])

for country, ids in video_ids_by_country.items():
    print(country, "unique video IDs:", len(ids))

# Compare US and GB, if both were loaded
us_ids = video_ids_by_country.get("US", set())
gb_ids = video_ids_by_country.get("GB", set())

print("Videos in both countries:", len(us_ids & gb_ids))
print("Videos only in US:", len(us_ids - gb_ids))
print("Videos only in GB:", len(gb_ids - us_ids))

# 4. Dictionary counter for channels
channel_counts = {}

for record in clean_records:
    channel = record.get("channel_title")

    if pd.isna(channel) or not str(channel).strip():
        channel = "Unknown"

    channel_counts[channel] = channel_counts.get(channel, 0) + 1

top_channels = sorted(
    channel_counts.items(),
    key=lambda item: item[1],
    reverse=True,
)[:15]

print("Top 15 channels:")
for channel, count in top_channels:
    print(channel, count)

# 5. Nested dictionary: country -> category -> count
country_category_counts = {}

for record in clean_records:
    country = record["country"]
    category_id = str(record.get("category_id", "Unknown"))

    country_category_counts.setdefault(country, {})
    country_category_counts[country][category_id] = (
        country_category_counts[country].get(category_id, 0) + 1
    )

print("Country-category counts:", country_category_counts)


Record example: {'video_id': 'g_N-w8LeAi8', 'trending_date': datetime.datetime(2017, 11, 14, 0, 0), 'title': 'Honest Football Prank LEAKED', 'channel_title': 'QuietSpeech10', 'category_id': 22, 'publish_time': '2017-11-14T07:04:28.000Z', 'tags': 'Monsoon|Robots', 'views': 48555, 'likes': 2189, 'dislikes': 180, 'comment_count': 136, 'thumbnail_link': 'https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about chess. Business enquiries in the channel about page.', 'country': 'US'}
First tuples: [('g_N-w8LeAi8', datetime.datetime(2017, 11, 14, 0, 0), 48555, 2189), ('Ugyql2NlPAd', datetime.datetime(2017, 11, 14, 0, 0), 198385, 1159), ('8fE19jkxJ9n', datetime.datetime(2017, 11, 14, 0, 0), 231656, 8742), ('WOY0nn7HCsO', datetime.datetime(2017, 11, 14, 0, 0), 8083, 176), ('2Lc_Cs5VVr_', datetime.datetime(2017, 11, 14, 0, 0), 373161, 11305)]
US unique

In [13]:

message = "I am a global variable"

def scope_demo():
    local_message = "I am a local variable"
    print("Inside function:", local_message)
    print("Global variable inside function:", message)

scope_demo()

print("Outside function:", message)

# This causes a NameError because local_message exists only
# inside scope_demo().
try:
    print(local_message)
except NameError as error:
    print("Expected error:", error)


Inside function: I am a local variable
Global variable inside function: I am a global variable
Outside function: I am a global variable
Expected error: name 'local_message' is not defined


In [14]:

def title_features(title):
    words = title.split()
    word_count = 0

    # Count words with a for loop.
    for word in words:
        word_count += 1

    # Count characters with a for loop.
    character_count = 0
    for character in title:
        character_count += 1

    # Use range(len(...)) and index access once.
    first_word = ""
    if len(words) > 0:
        first_word = words[0]

    indexed_characters = []
    for index in range(len(title)):
        indexed_characters.append(title[index])
        break  # Demonstrate indexed access once.

    return {
        "word_count": word_count,
        "character_count": character_count,
        "first_word": first_word,
        "first_indexed_character": (
            indexed_characters[0] if indexed_characters else ""
        ),
    }


print(title_features("Python for Data Analytics"))


{'word_count': 4, 'character_count': 25, 'first_word': 'Python', 'first_indexed_character': 'P'}


In [15]:

index = 0
first_popular_video = None

while index < len(clean_records):
    record = clean_records[index]
    index += 1

    if record["views"] <= VIEW_THRESHOLD:
        continue  # Skip videos below or equal to the threshold.

    first_popular_video = record
    break  # Stop when the first qualifying video is found.

if first_popular_video:
    print("First video above threshold:")
    print(first_popular_video["title"])
    print(f"Views: {first_popular_video['views']:,}")
else:
    print("No video found above the threshold.")


First video above threshold:
Official Cats Speech EXCLUSIVE
Views: 2,841,278


In [16]:

def classify_video(record):
    views = record.get("views", 0)

    # First: zero or invalid views.
    if views is None or pd.isna(views) or views <= 0:
        return "Invalid views"

    # Second: ratings disabled.
    ratings_disabled = record.get("ratings_disabled", False)

    if str(ratings_disabled).strip().lower() in ("true", "1"):
        return "Ratings disabled"

    likes = record.get("likes", 0)

    # Guard against missing or invalid likes.
    try:
        likes = int(likes)
        if likes < 0:
            likes = 0
    except (TypeError, ValueError, OverflowError):
        likes = 0

    like_ratio = likes / views

    # Nested if/elif/else classification.
    if views >= VIEW_THRESHOLD:
        if like_ratio >= LIKE_RATIO_THRESHOLD:
            return "High reach, strong engagement"
        elif like_ratio > 0:
            return "High reach, low engagement"
        else:
            return "High reach, no likes"

    else:
        if like_ratio >= LIKE_RATIO_THRESHOLD:
            return "Lower reach, strong engagement"
        elif like_ratio > 0:
            return "Lower reach, low engagement"
        else:
            return "Lower reach, no likes"


# Test with sample records.
sample_records = [
    {"views": 2_000_000, "likes": 100_000, "ratings_disabled": False},
    {"views": 2_000_000, "likes": 2_000, "ratings_disabled": False},
    {"views": 0, "likes": 0, "ratings_disabled": False},
    {"views": 500_000, "likes": 30_000, "ratings_disabled": True},
]

for sample in sample_records:
    print(classify_video(sample))


High reach, strong engagement
High reach, low engagement
Invalid views
Ratings disabled


record = {
    "views": 2_000_000,
    "likes": 100_000,
    "ratings_disabled": False
}

In [17]:

def safe_nonnegative_int(value, field_name, failures, default=None):
    """Convert a value to a non-negative integer.

    Returns None on failure when no default is supplied.
    Otherwise returns the provided default.
    """
    try:
        if pd.isna(value):
            raise ValueError("Missing value")

        number = int(value)

        if number < 0:
            raise ValueError("Negative value")

        return number

    except (TypeError, ValueError, OverflowError):
        failures[field_name] += 1
        return default


# Demonstrate the helper.
test_failures = {"views": 0, "likes": 0}

print(safe_nonnegative_int(1500, "views", test_failures))
print(safe_nonnegative_int(None, "likes", test_failures, default=0))
print("Failures:", test_failures)


1500
0
Failures: {'views': 0, 'likes': 1}
